In [2]:
import numpy as np
from scipy.fft import dct, idct, fft

np.random.seed(42)

In [3]:
def custom_dct(x: np.ndarray):
    """
    1. Reorder input
    2. Perform FFT
    3. Apply phase correction
    """
    N = len(x)
    y = np.zeros(N)

    for n in range(0, N//2):
        y[n] = x[2*n]

    for n in range(N//2, N):
        y[n] = x[2*N - 2*n - 1]

    Y = fft(y)

    """
    From Makhoul's paper on DCN, Eq (10): C(k) = 2Re [ W_{2N}^{k/2} \\sum{n=0}^{N-1} x(n) W_{2N}^{nk} ]
    where W_M = e^{-j2pi/M}. This expands to:    2Re [ e^{-j kpi/2N} \\sum{n=0}^{N-1} x(n) e^{-j 2nkpi/N} ]
    From DFT, Y_k = \\sum_{n=0}^{N-1} x(n) e^{-j 2nkpi/N}.
    When combined:                               2Re [ e^{-j kpi/2N} * Y_k ]

    e^{-j kpi/2N} * Y_k can be written as: e^{-j theta} * Y_k
    e^{-j theta} * Y_k = (cos theta - jsin theta) * (Yr + jYi) = 
    = (Yr * cos theta + Yi sin theta) + j(Yi cos theta - Yr sin theta)
    From 2Re [ e^{-j kpi/2N} * Y_k ], the imaginary part is discarded.

    In full: the final X becomes
    C(k) = 2 [ Y_k_Re * cos theta + Y_k_Im * sin theta ]
    """
    X = np.zeros(N)
    for k in range(N):
        theta = np.pi*k / (2*N)
        X[k] = 2*( Y[k].real * np.cos(theta) + Y[k].imag * np.sin(theta) )
    
    return X

In [ ]:
def custom_idct(X: np.ndarray):
    N = len(X) # malloc
    x = np.zeros(N) # malloc

    for n in range(N): # outer loop
        x[n] = 0 # malloc
        partial_sum = 0 # malloc

        for k in range(1, N): # inner loop
            theta = np.pi * ((2*n) + 1) * k / (2*N) # constant
            partial_sum += X[k] * np.cos(theta)

        x[n] = 1/N * ( (X[0]/2) + partial_sum )
    
    return x

In [5]:
x = np.random.rand(10)
x

array([0.37454012, 0.95071431, 0.73199394, 0.59865848, 0.15601864,
       0.15599452, 0.05808361, 0.86617615, 0.60111501, 0.70807258])

In [6]:
X = dct(x)
X

array([10.40273472,  0.26521523,  2.51800556, -1.36307904, -2.30476034,
       -0.01180947, -0.79666298, -1.01410762,  0.48472274, -1.57480758])

In [7]:
custom_dct(x)

array([10.40273472,  0.26521523,  2.51800556, -1.36307904, -2.30476034,
       -0.01180947, -0.79666298, -1.01410762,  0.48472274, -1.57480758])

In [8]:
idct(X)

array([0.37454012, 0.95071431, 0.73199394, 0.59865848, 0.15601864,
       0.15599452, 0.05808361, 0.86617615, 0.60111501, 0.70807258])

In [9]:
custom_idct(X)

array([0.37454012, 0.95071431, 0.73199394, 0.59865848, 0.15601864,
       0.15599452, 0.05808361, 0.86617615, 0.60111501, 0.70807258])